# 2a. Static Expert Parameters, Uniform Distribution

**Scenario**: Synthetic animals with fixed expert-level parameters,
uniform stimulus distribution. The simplest case.

**Tests**:
1. Model identification (GS + SBI)
2. Parameter recovery (GS + SBI) — can we get the true params back?
3. SBC calibration — are SBI posteriors well-calibrated?

In [ ]:
%matplotlib inline
import os, sys
from pathlib import Path

_NOTEBOOK_DIR = Path(os.path.abspath(''))
_NOTEBOOK_ROOT = _NOTEBOOK_DIR.parent
_PROJECT_ROOT = _NOTEBOOK_ROOT.parent
if str(_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT))
if str(_NOTEBOOK_ROOT) not in sys.path:
	sys.path.insert(0, str(_NOTEBOOK_ROOT))

from shared_setup import *

import time
import pickle
from analysis.validation import (
    make_synthetic_cohort,
    run_gs_model_id, run_sbi_model_id, summarise_agreement,
)

SBI_OK = False
try:
    import torch
    SBI_OK = True
    print(f'SBI available (torch={torch.__version__})')
except ImportError as e:
    print(f'SBI not available: {e}')

## 1. Configuration

In [ ]:
QUICK = True

if QUICK:
    N_SYNTHETIC = 3; N_GS_SEEDS = 2; BURN_IN = 500
    N_SBI_SIMS = 1_000; N_GENERIC_TRIALS = 300; N_CV_REPEATS = 4
    N_SBC_RUNS = 50; N_SBI_RECOVERY = 20
else:
    N_SYNTHETIC = 5; N_GS_SEEDS = 4; BURN_IN = 1000
    N_SBI_SIMS = 20_000; N_GENERIC_TRIALS = 2500; N_CV_REPEATS = 32
    N_SBC_RUNS = 500; N_SBI_RECOVERY = 100

SEED = 42; N_SESSIONS = 15; TRIALS_PER_SESSION = 350
print(f'Mode: {"QUICK" if QUICK else "FULL"}')

## 2. Generate Synthetic Animals

In [ ]:
animals = make_synthetic_cohort(
    n_per_model=N_SYNTHETIC, n_sessions=N_SESSIONS,
    trials_per_session=TRIALS_PER_SESSION,
    burn_in=BURN_IN, seed=SEED,
)
print(f'{len(animals)} animals ({N_SYNTHETIC} BE + {N_SYNTHETIC} SC)')
for a in animals:
    accs = [s.stats(['accuracy'])['accuracy'] for s in a['sessions']]
    print(f'  {a["animal_id"]}: {a["true_model"]} acc={np.mean(accs):.2f}')

## 3. Grid-Search: Model ID


In [ ]:
print('=== Grid-Search Model Identification ===')
gs_df = run_gs_model_id(animals, n_seeds=N_GS_SEEDS, burn_in=BURN_IN)
print(f'\nAccuracy: {gs_df["gs_correct"].sum()}/{len(gs_df)} '
      f'({gs_df["gs_correct"].mean():.0%})')

### 3b. Grid-Search Parameter Recovery

For correctly-identified animals: how close are the recovered
parameters to the true values?

In [ ]:
# ── GS parameter recovery scatter ─────────────────────────────────────────
for true_model in ['BE', 'SC']:
    sub_df = gs_df[(gs_df['true_model'] == true_model) & gs_df['gs_correct']]
    sub_animals = [a for a in animals
                   if a['animal_id'] in sub_df['animal_id'].values]

    if len(sub_animals) < 2:
        print(f'{true_model}: too few correct recoveries for scatter')
        continue

    param_names = sub_animals[0]['true_params'].get_param_names()
    n_params = len(param_names)

    fig, axes = plt.subplots(1, n_params, figsize=(4 * n_params, 4))
    if n_params == 1: axes = [axes]

    for ax, pname in zip(axes, param_names):
        true_vals = [getattr(a['true_params'], pname) for a in sub_animals]
        rec_vals = []
        for a in sub_animals:
            row = sub_df[sub_df['animal_id'] == a['animal_id']].iloc[0]
            rp = row.get('gs_recovered_params', {})
            rec_vals.append(rp.get(pname, rp.get('sigma_noise' if pname == 'sigma_percep' else pname, np.nan)))

        ax.scatter(true_vals, rec_vals, alpha=0.7, s=50)
        lims = [min(min(true_vals), min(rec_vals)) * 0.9,
                max(max(true_vals), max(rec_vals)) * 1.1]
        ax.plot(lims, lims, '--', color='grey', alpha=0.5)
        ax.set_xlabel(f'True {pname}')
        ax.set_ylabel(f'Recovered {pname}')
        ax.set_title(pname)
        ax.set_aspect('equal')

    fig.suptitle(f'{true_model}: GS Parameter Recovery (n={len(sub_animals)})')
    plt.tight_layout()
    plt.show()

## 4. SBI: Model ID

In [ ]:
sbi_df = pd.DataFrame()
be_snpe, sc_snpe = None, None

if SBI_OK:
    print('=== SBI Model Identification ===')
    sbi_df, be_snpe, sc_snpe = run_sbi_model_id(
        animals, n_sbi_sims=N_SBI_SIMS,
        n_generic_trials=N_GENERIC_TRIALS,
        n_cv_repeats=N_CV_REPEATS, burn_in=BURN_IN, seed=SEED,
        return_networks=True,
    )
    print(f'\nAccuracy: {sbi_df["sbi_correct"].sum()}/{len(sbi_df)} '
          f'({sbi_df["sbi_correct"].mean():.0%})')
else:
    print('SBI not available')

### 4b. SBI Parameter Recovery

Sample from prior → simulate → condition posterior → extract median.
Compare median to true values.

In [ ]:
if SBI_OK and be_snpe is not None:
    from inference.diagnostics import (
        parameter_recovery, plot_recovery_scatter,
    )

    for mt, snpe in [('BE', be_snpe), ('SC', sc_snpe)]:
        print(f'\n=== {mt} SBI Recovery ===')
        try:
            rec = parameter_recovery(
                posterior=snpe['posterior'],
                simulator=snpe['sbi_sim'],
                prior=snpe['prior'],
                n_recoveries=N_SBI_RECOVERY,
                n_posterior_samples=200,
                seed=SEED,
                param_names=snpe['param_names'],
            )
            fig = plot_recovery_scatter(rec, title=f'{mt} SBI Recovery')
            plt.show()
        except Exception as e:
            print(f'  Failed: {e}')

### 4c. SBC Calibration

Rank histograms: if the posterior is well-calibrated, ranks should
be uniformly distributed.

In [ ]:
if SBI_OK and be_snpe is not None:
    from inference.diagnostics import run_sbc, plot_sbc_ranks

    for mt, snpe in [('BE', be_snpe), ('SC', sc_snpe)]:
        print(f'\n=== {mt} SBC ===')
        try:
            sbc = run_sbc(
                posterior=snpe['posterior'],
                simulator=snpe['sbi_sim'],
                prior=snpe['prior'],
                n_sbc_runs=N_SBC_RUNS,
                n_posterior_samples=200,
                seed=SEED,
                param_names=snpe['param_names'],
            )
            fig = plot_sbc_ranks(sbc, title=f'{mt} SBC Rank Histograms')
            plt.show()
        except Exception as e:
            print(f'  Failed: {e}')

## 5. Agreement

In [ ]:
merged = summarise_agreement(gs_df, sbi_df, 'Static Uniform')
print()
print(merged.to_string(index=False))

## 6. Save

In [ ]:
save_path = Path('../results/validation/2a_static_uniform.pkl')
save_path.parent.mkdir(parents=True, exist_ok=True)
with open(save_path, 'wb') as f:
    pickle.dump({'gs': gs_df, 'sbi': sbi_df if SBI_OK else None,
                 'scenario': '2a_static_uniform'}, f)
print(f'Saved to {save_path}')